## Housing Prices Estimator

## 1. Problem Statement 
A simple yet challenging project, to predict the housing price based on certain factors like house area, bedrooms, furnished, nearness to mainroad, etc. The dataset is small yet, it's complexity arises due to the fact that it has strong multicollinearity. Can you overcome these obstacles & build a decent predictive model?

## 2. Data Collection

The dataset is Collected from: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset

License: CC0: Public Domain

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("dataset\Housing.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## 3. Data Engineering

### First and Foremost
- find a count of MISSING value column wise 
- data types of the column

In [3]:
df.isnull().sum() # There is NO missing values.

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [4]:
print(df.dtypes)
type(df.dtypes) # Series object


price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object


pandas.core.series.Series

### Now let's see the count unique values present in each column.

In [5]:
print(df.nunique())
type(df.nunique()) # Series object

price               219
area                284
bedrooms              6
bathrooms             4
stories               4
mainroad              2
guestroom             2
basement              2
hotwaterheating       2
airconditioning       2
parking               4
prefarea              2
furnishingstatus      3
dtype: int64


pandas.core.series.Series

Let's make a custom function to get number of unique values and data types of each column as a dataframe

In [6]:
# We can combine SERIES object as key-value dict and make a dataframe using pandas
def getDType_Unique(df):
    customdf=pd.DataFrame({"datatype":df.dtypes,"unique_values":df.nunique()})
    print(customdf)

In [7]:
getDType_Unique(df)

                 datatype  unique_values
price               int64            219
area                int64            284
bedrooms            int64              6
bathrooms           int64              4
stories             int64              4
mainroad           object              2
guestroom          object              2
basement           object              2
hotwaterheating    object              2
airconditioning    object              2
parking             int64              4
prefarea           object              2
furnishingstatus   object              3


##### We are getting all the unique values under both categorical and numeric feature, looking for if there is any typo of scope of feature reduction {we do not want highly correlated features}

In [8]:
print("Following are list of all unique values under each Categorical Column")
print(df["mainroad"].unique().tolist())
print(df["guestroom"].unique().tolist())
print(df["basement"].unique().tolist())
print(df["hotwaterheating"].unique().tolist())
print(df["airconditioning"].unique().tolist())
print(df["prefarea"].unique().tolist())
print(df["furnishingstatus"].unique().tolist())

Following are list of all unique values under each Categorical Column
['yes', 'no']
['no', 'yes']
['no', 'yes']
['no', 'yes']
['yes', 'no']
['yes', 'no']
['furnished', 'semi-furnished', 'unfurnished']


In [9]:
print("Following are list of all unique values under each Numeric Column")
print(df["bedrooms"].unique().tolist())
print(df["bathrooms"].unique().tolist())
print(df["stories"].unique().tolist())
print(df["parking"].unique().tolist())

Following are list of all unique values under each Numeric Column
[4, 3, 5, 2, 6, 1]
[2, 4, 1, 3]
[3, 4, 2, 1]
[2, 3, 0, 1]


## 4. Let us do our train-test Split [target column = price]

In [10]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


InDependent Fatures goes in X, so let us drop price

In [11]:
X = df.drop("price", axis=1)
X.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


Our Dependent Feature aka Target variable, goes into y

In [12]:
y = df["price"]
y.head()

0    13300000
1    12250000
2    12250000
3    12215000
4    11410000
Name: price, dtype: int64

In [13]:
print(X.shape)
print(y.shape)

(545, 12)
(545,)


In [14]:
from sklearn.model_selection import  train_test_split

# Train test split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [15]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(436, 12)
(436,)
(109, 12)
(109,)


## 5. Data Preprocessing

### Let us segregate our Discrete and Continious Values

But Before that we need to seperate categorical and numeric values

In [16]:
cat_features = X.select_dtypes("object").columns.tolist() # Contains categorical features
cat_features

['mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'prefarea',
 'furnishingstatus']

In [17]:
num_features = X.select_dtypes(exclude="object").columns.tolist() # Contains numerical features
num_features

['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

#### Let us verify the columns by using our custom method

In [18]:
getDType_Unique(X)

                 datatype  unique_values
area                int64            284
bedrooms            int64              6
bathrooms           int64              4
stories             int64              4
mainroad           object              2
guestroom          object              2
basement           object              2
hotwaterheating    object              2
airconditioning    object              2
parking             int64              4
prefarea           object              2
furnishingstatus   object              3


### Now we will use OHE to covert categorical data to numerical and StandardScaler to scale the numeric data into a normal distribution

In [19]:
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import  ColumnTransformer

# OHE is for categorical features
# Standard Scaler is for scaling down data in same scale (although not necessary here)
# Column Transformer helps to build a pipeline to apply multiple transformations.

numeric_transformer= StandardScaler()
oh_transformer = OneHotEncoder(drop="first") # As to represet 3 features, 2 columns is enough

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoding", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features )
    ], remainder="passthrough"
)

Now we fit_transform our Train Data and overwrite it and transform our Test Data and overwrite it.

In [20]:
## applying fit_transform for training dataset
X_train = preprocessor.fit_transform(X_train)

In [21]:
# X_train.head() wont work as X_train is no more  a dataframe
pd.DataFrame(X_train).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.384168,0.055271,1.539173,2.587644,0.367957
1,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.929181,0.055271,1.539173,-0.912499,2.709987
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.607755,-1.283514,-0.557950,-0.912499,1.538972
3,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,-1.155492,0.055271,-0.557950,0.254215,-0.803059
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.637730,0.055271,-0.557950,0.254215,-0.803059


In [22]:
# Apply same transformation for X_test

X_test = preprocessor.transform(X_test)

## 6. Model Training: Apply Random Forest Regressor and Other Models for Comparision

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import  LinearRegression, Ridge,Lasso
from sklearn.neighbors import  KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import  r2_score,mean_absolute_error, mean_squared_error



We will create a custom function to evaluate the models

In [24]:
def evalMod(tr, pred):
    mae=mean_absolute_error(tr,pred)
    mse=mean_squared_error(tr,pred)
    rmse=np.sqrt(mean_squared_error(tr,pred))
    r2_square=r2_score(tr,pred)
    return mae,rmse,r2_square

In [25]:
# Creating a dict of models

models = {
    "Linear Regression": LinearRegression(),
    "Lasso":Lasso(),
    "Ridge": Ridge(),
    "K-Neighbours Regression": KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest":RandomForestRegressor()
}

In [26]:
models_stats={}
for name,model in models.items():
    model.fit(X_train,y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate model.

    model_train_mae,model_train_rmse,model_train_r2_square=evalMod(y_train, y_train_pred)

    model_test_mae,model_test_rmse,model_test_r2_square=evalMod(y_test, y_test_pred)

    models_stats[name]={"Training":[model_train_mae,model_train_rmse,model_train_r2_square], "Testing": [model_test_mae,model_test_rmse,model_test_r2_square]}



In [27]:
models_stats

{'Linear Regression': {'Training': [np.float64(719242.8936724713),
   np.float64(984051.9236507412),
   0.6859438988560158],
  'Testing': [np.float64(970043.4039201641),
   np.float64(1324506.96009144),
   0.6529242642153177]},
 'Lasso': {'Training': [np.float64(719242.3172851637),
   np.float64(984051.9236794699),
   0.6859438988376786],
  'Testing': [np.float64(970044.0921120684),
   np.float64(1324508.1604940866),
   0.6529236351045118]},
 'Ridge': {'Training': [np.float64(718437.6573658905),
   np.float64(984097.7712962645),
   0.6859146340014346],
  'Testing': [np.float64(971370.2374030841),
   np.float64(1326096.0986867456),
   0.6520909242049433]},
 'K-Neighbours Regression': {'Training': [np.float64(614085.6605504587),
   np.float64(892233.7390940671),
   0.7418164970074341],
  'Testing': [np.float64(1031569.8348623853),
   np.float64(1457128.6922289731),
   0.5799397530885937]},
 'Decision Tree': {'Training': [np.float64(8107.798165137615),
   np.float64(67088.47540457372),
  

In [28]:
def printModelPerformance(modelstats):
    for model_name, stats in modelstats.items():
        print(f"{model_name} Statistics")
        
        # Loop over 'Training' and 'Testing' sets
        for set_name, metrics in stats.items():
            print(f"Model {set_name} Performance Set")
            print("- Mean Absolute Error: ", round(metrics[0], 3))
            print("- Root Mean Square Error: ", round(metrics[1], 3))
            print("- R2 Score: ", round(metrics[2], 3))
            print("-----------------------------------------------------------")

        
        print("\n")

In [29]:
printModelPerformance(modelstats=models_stats)

Linear Regression Statistics
Model Training Performance Set
- Mean Absolute Error:  719242.894
- Root Mean Square Error:  984051.924
- R2 Score:  0.686
-----------------------------------------------------------
Model Testing Performance Set
- Mean Absolute Error:  970043.404
- Root Mean Square Error:  1324506.96
- R2 Score:  0.653
-----------------------------------------------------------


Lasso Statistics
Model Training Performance Set
- Mean Absolute Error:  719242.317
- Root Mean Square Error:  984051.924
- R2 Score:  0.686
-----------------------------------------------------------
Model Testing Performance Set
- Mean Absolute Error:  970044.092
- Root Mean Square Error:  1324508.16
- R2 Score:  0.653
-----------------------------------------------------------


Ridge Statistics
Model Training Performance Set
- Mean Absolute Error:  718437.657
- Root Mean Square Error:  984097.771
- R2 Score:  0.686
-----------------------------------------------------------
Model Testing Perfor

#### As We Can see Deicison Tree and Random Forest are doing very good in Training Set wrt other Models but failing in Test, maybe because of Overfitting; so let us try to Hyper Parameter Tune them.

## 7. Hyper Parameter Tuning

In [30]:
decision_tree_params = {
    'criterion':["squared_error", "friedman_mse", "absolute_error", "poisson"],
    'splitter':["best","random"],
    'max_depth':[5,8,15,None,10],
    'max_features':["sqrt","log2"]
}

random_forest_params= {
    'n_estimators': [100,200,500,1000],
    'criterion': ["squared_error", "friedman_mse", "poisson"],
    'max_depth':[5,8,15,None,10],
    'max_features':["sqrt","log2",None]
}

In [31]:
randomcv_models = [("DT",DecisionTreeRegressor(),decision_tree_params),
                    ("RF",RandomForestRegressor(),random_forest_params)]

In [ ]:
from sklearn.model_selection import  RandomizedSearchCV

model_params={}

for name,model,param in randomcv_models:
    random=RandomizedSearchCV(estimator=model,param_distributions=param,n_iter=100,verbose=10,n_jobs=-1,cv=3)

    random.fit(X_train,y_train)

    model_params[name]=random.best_params_

In [33]:
for model_name in model_params:
    print(f"------------------- Best Params for {model_name} -------------------")
    print(model_params[model_name])

------------------- Best Params for DT -------------------
{'splitter': 'best', 'max_features': 'sqrt', 'max_depth': 5, 'criterion': 'friedman_mse'}


## 8. Retraining using the Hyperparameters:

In [34]:
models = {
    "Decision Tree":DecisionTreeRegressor(splitter= 'best', max_features='sqrt', max_depth= 5, criterion= 'absolute_error'),
    "Random Forest":RandomForestRegressor(n_estimators= 1000, max_features= 'log2', max_depth= 8, criterion='friedman_mse')
}

In [35]:
models_stats={}
for name,model in models.items():
    model.fit(X_train,y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate model.

    model_train_mae,model_train_rmse,model_train_r2_square=evalMod(y_train, y_train_pred)

    model_test_mae,model_test_rmse,model_test_r2_square=evalMod(y_test, y_test_pred)

    models_stats[name]={"Training":[model_train_mae,model_train_rmse,model_train_r2_square], "Testing": [model_test_mae,model_test_rmse,model_test_r2_square]}

In [36]:
printModelPerformance(modelstats=models_stats)

Decision Tree Statistics
Model Training Performance Set
- Mean Absolute Error:  748404.679
- Root Mean Square Error:  1114454.152
- R2 Score:  0.597
-----------------------------------------------------------
Model Testing Performance Set
- Mean Absolute Error:  1216715.596
- Root Mean Square Error:  1718935.981
- R2 Score:  0.415
-----------------------------------------------------------


Random Forest Statistics
Model Training Performance Set
- Mean Absolute Error:  442795.614
- Root Mean Square Error:  599180.346
- R2 Score:  0.884
-----------------------------------------------------------
Model Testing Performance Set
- Mean Absolute Error:  1017209.042
- Root Mean Square Error:  1390535.613
- R2 Score:  0.617
-----------------------------------------------------------




### We can See Significant Performace Improvement of RF wrt DT